In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="11-people-you-may-know",
    notebook_name="03_ranking_and_privacy.ipynb"
)

# 03 -- Ranking, Privacy, and Production Challenges for PYMK

**Goal:** Cover the ranking pipeline, multi-objective optimization, privacy considerations, cold-start, and serving architecture for People You May Know.

---

## The School Dance Analogy

Imagine you are organizing a school dance and you need to suggest dance partners to everyone.

- **Ranking:** Who is the best match for each person? You need to score and sort.
- **Multi-objective:** You want partners who are relevant AND diverse (not all from the same friend group) AND reciprocal (both should want to dance together).
- **Privacy:** Some kids said "do NOT suggest me to this person" -- you must respect that.
- **Notifications:** You need to carefully decide WHEN and HOW to tell people about their suggestions.
- **Cold start:** A new kid just transferred in -- you know nothing about them.
- **Serving:** You need to compute suggestions for 1,000 students efficiently.

These are exactly the challenges in a production PYMK system.

## 1. Ranking Candidates (Pointwise LTR)

After FoF candidate generation gives us ~1M candidates per user, we need to **score and rank** them.

### The Scoring Function

For each (user, candidate) pair, we compute:

$$\text{score}(u, v) = f(\text{graph\_features}, \text{profile\_features}, \text{affinity\_features})$$

Where f is our trained model (GNN, GBDT, or neural network).

### Two-Stage Ranking

```
1B users -> FoF Service -> ~1M candidates -> Lightweight Ranker -> ~1000 -> Full Model -> Top 50
                           (2-hop BFS)        (simple features)              (all features)
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import random
import math

np.random.seed(42)
random.seed(42)

# === Simulate a ranking pipeline ===

class PYMKRanker:
    """
    Two-stage ranking pipeline for PYMK.
    
    12-year-old version:
    Stage 1 (Fast): Quickly check the basics -- how many mutual friends?
                     Same school? Same company? Throw away obviously bad matches.
    Stage 2 (Precise): For the remaining good candidates, do a thorough
                       analysis with all available features.
    
    Staff-level detail:
    Stage 1 uses a lightweight model (logistic regression on graph features)
    with O(1) feature computation per candidate.
    Stage 2 uses the full GNN or deep model with all features.
    This 2-stage approach reduces latency from O(1M * full_model) to
    O(1M * lightweight + 1K * full_model).
    """
    
    def __init__(self):
        pass
    
    def stage1_lightweight_score(self, common_neighbors, same_company, same_school):
        """Fast scoring with simple features. <0.1ms per candidate."""
        return 0.5 * common_neighbors + 2.0 * same_company + 1.5 * same_school
    
    def stage2_full_score(self, features):
        """Full model scoring with all features. ~1ms per candidate."""
        # Simulate a complex model score
        score = (
            0.3 * features['common_neighbors'] / 10.0
            + 0.2 * features['jaccard']
            + 0.15 * features['adamic_adar'] / 5.0
            + 0.1 * features['same_company']
            + 0.1 * features['same_school']
            + 0.05 * features['same_city']
            + 0.05 * features['profile_visits'] / 10.0
            + 0.05 * features['time_discounted_cn'] / 5.0
        )
        return min(score, 1.0)


# Simulate candidates for a user
n_candidates = 1000

candidates = []
for i in range(n_candidates):
    cn = max(0, int(np.random.exponential(3)))
    same_company = int(random.random() < 0.1)
    same_school = int(random.random() < 0.15)
    same_city = int(random.random() < 0.3)
    
    candidates.append({
        'candidate_id': i,
        'common_neighbors': cn,
        'jaccard': round(random.uniform(0, 0.3) * (cn > 0), 4),
        'adamic_adar': round(cn * random.uniform(0.5, 2.0), 4) if cn > 0 else 0,
        'same_company': same_company,
        'same_school': same_school,
        'same_city': same_city,
        'profile_visits': np.random.poisson(1) if random.random() < 0.2 else 0,
        'time_discounted_cn': round(cn * random.uniform(0.3, 1.0), 4) if cn > 0 else 0,
    })

ranker = PYMKRanker()

# Stage 1: Lightweight scoring
for c in candidates:
    c['stage1_score'] = ranker.stage1_lightweight_score(
        c['common_neighbors'], c['same_company'], c['same_school'])

# Keep top 100 from stage 1
stage1_top = sorted(candidates, key=lambda x: -x['stage1_score'])[:100]

# Stage 2: Full scoring on top 100
for c in stage1_top:
    c['stage2_score'] = ranker.stage2_full_score(c)

final_top = sorted(stage1_top, key=lambda x: -x['stage2_score'])[:10]

print("TWO-STAGE RANKING PIPELINE")
print("=" * 60)
print(f"FoF candidates:  {n_candidates}")
print(f"After Stage 1:   {len(stage1_top)} (lightweight scoring)")
print(f"After Stage 2:   {len(final_top)} (full model scoring)")
print(f"\nTop 10 Final Suggestions:")
print(f"{'Rank':>4} {'ID':>4} {'CN':>4} {'Company':>8} {'School':>7} {'S1 Score':>9} {'S2 Score':>9}")
print("-" * 50)
for i, c in enumerate(final_top):
    print(f"{i+1:>4} {c['candidate_id']:>4} {c['common_neighbors']:>4} "
          f"{'Yes' if c['same_company'] else 'No':>8} "
          f"{'Yes' if c['same_school'] else 'No':>7} "
          f"{c['stage1_score']:>9.2f} {c['stage2_score']:>9.4f}")

## 2. Multi-Objective Optimization

In production, we do not just optimize for "most likely to connect." We balance multiple objectives:

| Objective | Why It Matters | Example |
|-----------|---------------|--------|
| **Relevance** | Suggest people the user actually wants to connect with | High common neighbors |
| **Diversity** | Don't show 10 people from the same company | Mix departments, schools |
| **Reciprocity** | Both sides should want to connect | User A visited User B's profile |
| **Freshness** | Show new candidates, not the same ones every day | Rotate suggestions |
| **Novelty** | Introduce unexpected but valuable connections | Cross-department |

In [ ]:
def rerank_for_diversity(candidates, top_k=10, diversity_weight=0.3):
    """
    Re-rank candidates to balance relevance and diversity.
    
    12-year-old version:
    If the top 10 candidates are ALL from your soccer team, that is
    boring! You want some soccer friends, some chess club friends,
    some neighbors, and some classmates. This function makes sure
    the suggestions are a healthy mix.
    
    Staff-level detail:
    Uses a greedy MMR (Maximal Marginal Relevance) approach.
    At each step, pick the candidate that maximizes:
    score = (1-lambda) * relevance - lambda * max_similarity_to_selected
    """
    selected = []
    remaining = candidates.copy()
    
    for _ in range(min(top_k, len(remaining))):
        best_score = -float('inf')
        best_idx = 0
        
        for i, cand in enumerate(remaining):
            relevance = cand['stage2_score']
            
            # Compute similarity to already selected candidates
            if selected:
                similarities = []
                for sel in selected:
                    sim = (
                        int(cand['same_company'] == sel['same_company'] and cand['same_company'] == 1) * 0.5
                        + int(cand['same_school'] == sel['same_school'] and cand['same_school'] == 1) * 0.3
                        + int(cand['same_city'] == sel['same_city'] and cand['same_city'] == 1) * 0.2
                    )
                    similarities.append(sim)
                max_sim = max(similarities)
            else:
                max_sim = 0
            
            mmr_score = (1 - diversity_weight) * relevance - diversity_weight * max_sim
            
            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = i
        
        selected.append(remaining.pop(best_idx))
    
    return selected


# Compare: pure relevance vs diversity-aware ranking
pure_relevance = sorted(stage1_top, key=lambda x: -x['stage2_score'])[:10]
diverse = rerank_for_diversity(stage1_top[:50], top_k=10, diversity_weight=0.3)

print("PURE RELEVANCE vs DIVERSITY-AWARE RANKING")
print("=" * 60)

for label, results in [('Pure Relevance', pure_relevance), ('Diversity-Aware', diverse)]:
    n_company = sum(1 for c in results if c['same_company'])
    n_school = sum(1 for c in results if c['same_school'])
    n_city = sum(1 for c in results if c['same_city'])
    avg_score = np.mean([c['stage2_score'] for c in results])
    
    print(f"\n{label}:")
    print(f"  Avg relevance score:  {avg_score:.4f}")
    print(f"  Same company:         {n_company}/10")
    print(f"  Same school:          {n_school}/10")
    print(f"  Same city:            {n_city}/10")
    unique_attrs = len(set(
        (c['same_company'], c['same_school'], c['same_city']) for c in results))
    print(f"  Unique attribute combos: {unique_attrs}")

print(f"\nDiversity-aware ranking trades a small amount of relevance")
print(f"for a much more varied set of suggestions.")

## 3. Privacy Considerations

Privacy in PYMK is a minefield. Get it wrong and you make headlines for the wrong reasons.

### Privacy Requirements

| Feature | Privacy Rule | Why |
|---------|-------------|-----|
| **Profile views** | NEVER reveal who viewed your profile | "Stalker" signal is creepy |
| **Blocked users** | NEVER suggest blocked users | User explicitly said no |
| **Phone contacts** | Only use with explicit consent | Cambridge Analytica lesson |
| **Location data** | Use coarsely (city level, not GPS) | Prevents physical tracking |
| **Search queries** | Don't reveal "User searched for you" | Feels invasive |
| **Ex-connections** | Don't suggest recently disconnected users | They disconnected for a reason |

In [ ]:
class PrivacyFilter:
    """
    Privacy-aware filtering for PYMK suggestions.
    
    12-year-old version:
    Before showing any suggestions, we check a list of rules:
    - Did this person block the other? REMOVE.
    - Did they recently UNfriend each other? REMOVE.
    - Is the account too new (possibly spam)? REMOVE.
    - Is the person in a restricted visibility mode? REMOVE.
    
    Staff-level detail:
    Privacy filters run AFTER scoring but BEFORE display.
    They must be applied consistently across all surfaces
    (web, mobile, email notifications, API).
    Violations can result in regulatory fines (GDPR, CCPA).
    """
    
    def __init__(self):
        self.blocked_pairs = set()  # (blocker, blocked)
        self.hidden_users = set()   # Users who opted out of PYMK
        self.recently_disconnected = set()  # (user_a, user_b)
        self.spam_accounts = set()
    
    def add_block(self, blocker, blocked):
        self.blocked_pairs.add((blocker, blocked))
    
    def hide_user(self, user_id):
        self.hidden_users.add(user_id)
    
    def add_recent_disconnect(self, user_a, user_b):
        self.recently_disconnected.add((min(user_a, user_b), max(user_a, user_b)))
    
    def mark_spam(self, user_id):
        self.spam_accounts.add(user_id)
    
    def filter_candidates(self, user_id, candidates):
        """Apply all privacy filters. Returns filtered list with reasons."""
        filtered = []
        removed = []
        
        for cand in candidates:
            cid = cand['candidate_id']
            
            # Check blocks (bidirectional)
            if (user_id, cid) in self.blocked_pairs or (cid, user_id) in self.blocked_pairs:
                removed.append((cid, 'BLOCKED'))
                continue
            
            # Check hidden users
            if cid in self.hidden_users:
                removed.append((cid, 'HIDDEN_PROFILE'))
                continue
            
            # Check recent disconnections
            pair = (min(user_id, cid), max(user_id, cid))
            if pair in self.recently_disconnected:
                removed.append((cid, 'RECENTLY_DISCONNECTED'))
                continue
            
            # Check spam
            if cid in self.spam_accounts:
                removed.append((cid, 'SPAM_ACCOUNT'))
                continue
            
            filtered.append(cand)
        
        return filtered, removed


# Demo
privacy = PrivacyFilter()
privacy.add_block(0, 5)   # User 0 blocked candidate 5
privacy.hide_user(12)     # Candidate 12 opted out
privacy.add_recent_disconnect(0, 8)  # User 0 and 8 recently disconnected
privacy.mark_spam(20)     # Candidate 20 is spam

# Apply filters to our ranked candidates
filtered, removed = privacy.filter_candidates(0, final_top)

print("PRIVACY FILTERING")
print("=" * 50)
print(f"Candidates before filtering: {len(final_top)}")
print(f"Candidates after filtering:  {len(filtered)}")
print(f"Removed: {len(removed)}")
for cid, reason in removed:
    print(f"  Candidate {cid}: {reason}")

print(f"\nPrivacy rules that interviewers love to hear about:")
print(f"  1. NEVER reveal profile viewers in PYMK explanations")
print(f"  2. ALWAYS respect blocks bidirectionally")
print(f"  3. Respect 'do not suggest' preferences (GDPR right to object)")
print(f"  4. Don't use phone contacts without explicit consent")
print(f"  5. Age-gate: extra restrictions for minors")

## 4. Notification Strategies

PYMK suggestions can be surfaced through multiple channels:

| Channel | When to Use | Example |
|---------|------------|--------|
| **In-app (homepage)** | Every visit | "People you may know" widget |
| **Push notification** | High-confidence only | "3 people from Google just joined" |
| **Email digest** | Weekly | "You have 5 new connection suggestions" |
| **Post-action** | After profile view | "Since you viewed Alice's profile..." |

### Notification Design Principles
1. **Threshold:** Only notify if score > high threshold (avoid spam)
2. **Rate limit:** Max 1 push notification per day
3. **Batch:** Group multiple suggestions in one notification
4. **Timing:** Send at the user's typical active hours
5. **Respect opt-out:** Honor notification preferences

## 5. Cold-Start Problem

### The Challenge

New users have:
- Zero connections (no graph features)
- Zero interaction history (no behavioral features)
- Only profile information (school, company, city)

### Solutions

In [ ]:
class ColdStartHandler:
    """
    Handle cold-start users in PYMK.
    
    12-year-old version:
    A new kid joins your school. You don't know who their friends
    are yet, but you DO know:
    - They transferred from Lincoln Middle School (suggest other Lincoln kids)
    - They live on Oak Street (suggest neighborhood kids)
    - They play soccer (suggest soccer team members)
    
    Staff-level detail:
    Cold-start strategies in order of priority:
    1. Phone contacts (with consent) -> highest quality signal
    2. Same school/company -> attribute matching
    3. Same city/region -> geographic proximity
    4. Popular users -> exploration via bandits
    
    As the user forms connections, we gradually transition from
    content-based features to graph-based features.
    """
    
    def __init__(self, all_users_df):
        self.users = all_users_df
    
    def get_cold_start_suggestions(self, new_user_profile, top_k=10):
        """Generate PYMK for a user with zero connections."""
        candidates = []
        
        for _, existing_user in self.users.iterrows():
            score = 0.0
            reasons = []
            
            # School match (strongest signal for cold-start)
            if new_user_profile['school'] == existing_user['school']:
                score += 3.0
                reasons.append('Same school')
            
            # Company match
            if new_user_profile['company'] == existing_user['company']:
                score += 2.5
                reasons.append('Same company')
            
            # Industry match
            if new_user_profile['industry'] == existing_user['industry']:
                score += 1.0
                reasons.append('Same industry')
            
            # City match
            if new_user_profile['city'] == existing_user['city']:
                score += 0.5
                reasons.append('Same city')
            
            # Popularity bonus (more connections = more likely to accept)
            popularity_bonus = min(existing_user['connections'] / 100.0, 0.5)
            score += popularity_bonus
            
            if score > 0:
                candidates.append({
                    'user_id': existing_user['user_id'],
                    'score': score,
                    'reasons': reasons,
                    'company': existing_user['company'],
                    'school': existing_user['school'],
                })
        
        # Sort by score
        candidates.sort(key=lambda x: -x['score'])
        return candidates[:top_k]


# Simulate existing users
np.random.seed(42)
companies = ['Google', 'Meta', 'Amazon', 'Apple', 'Microsoft']
schools = ['MIT', 'Stanford', 'CMU', 'Berkeley', 'Harvard']
industries = ['Tech', 'Finance', 'Healthcare']
cities = ['San Francisco', 'New York', 'Seattle']

existing_users = pd.DataFrame({
    'user_id': range(50),
    'company': [random.choice(companies) for _ in range(50)],
    'school': [random.choice(schools) for _ in range(50)],
    'industry': [random.choice(industries) for _ in range(50)],
    'city': [random.choice(cities) for _ in range(50)],
    'connections': [random.randint(5, 200) for _ in range(50)],
})

# New user just joined
new_user = {
    'school': 'MIT',
    'company': 'Google',
    'industry': 'Tech',
    'city': 'San Francisco',
}

handler = ColdStartHandler(existing_users)
suggestions = handler.get_cold_start_suggestions(new_user, top_k=10)

print("COLD-START SUGGESTIONS")
print("=" * 60)
print(f"New user: School={new_user['school']}, Company={new_user['company']}, "
      f"City={new_user['city']}")
print(f"\n{'Rank':>4} {'ID':>4} {'Score':>6} {'Company':>10} {'School':>10} {'Reasons'}")
print("-" * 60)
for i, s in enumerate(suggestions):
    print(f"{i+1:>4} {s['user_id']:>4} {s['score']:>6.1f} {s['company']:>10} "
          f"{s['school']:>10} {', '.join(s['reasons'])}")

print(f"\nCold-start priority:")
print(f"  1. Same school + same company (strongest signal)")
print(f"  2. Same school OR same company")
print(f"  3. Same industry + same city")
print(f"  4. Popular users in same city (exploration)")

## 6. Serving Architecture

### Why Batch Prediction?

With 1B users, we cannot compute PYMK in real-time for everyone. Social graphs are stable (your friends list does not change every minute), so pre-computing works well.

### The Two-Pipeline Architecture

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('PYMK System Architecture: Two Pipelines', fontsize=16, fontweight='bold')

# Pipeline 1: Generation (Offline/Batch)
rect1 = FancyBboxPatch((0.5, 5.5), 13, 4, boxstyle='round,pad=0.15',
                        facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(rect1)
ax.text(7, 9.0, 'Pipeline 1: PYMK GENERATION (Offline/Batch)', 
        ha='center', fontsize=13, fontweight='bold', color='#1565C0')

gen_steps = [
    (1.0, 6.5, 'For Each\nActive User', '#FFE0B2'),
    (3.5, 6.5, 'FoF Service\n(2-hop BFS)\n1B -> 1M', '#B3E5FC'),
    (6.0, 6.5, 'Lightweight\nRanker\n1M -> 1K', '#C8E6C9'),
    (8.5, 6.5, 'Full Model\n(GNN/GBDT)\n1K -> 50', '#F8BBD0'),
    (11.0, 6.5, 'Store in\nDatabase', '#FFF9C4'),
]

for x, y, text, color in gen_steps:
    box = FancyBboxPatch((x, y), 2, 1.8, boxstyle='round,pad=0.1',
                        facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + 1, y + 0.9, text, ha='center', va='center', fontsize=8, fontweight='bold')

for x1, x2 in [(3.0, 3.5), (5.5, 6.0), (8.0, 8.5), (10.5, 11.0)]:
    ax.annotate('', xy=(x2, 7.4), xytext=(x1, 7.4),
               arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Pipeline 2: Serving (Online)
rect2 = FancyBboxPatch((0.5, 0.5), 13, 4.2, boxstyle='round,pad=0.15',
                        facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(rect2)
ax.text(7, 4.3, 'Pipeline 2: ONLINE SERVING', 
        ha='center', fontsize=13, fontweight='bold', color='#2E7D32')

serve_steps = [
    (1.0, 1.5, 'User Opens\nApp', '#FFE0B2'),
    (3.5, 1.5, 'Check\nDatabase', '#B3E5FC'),
    (6.5, 2.5, 'Pre-computed\nPYMK Found!\n-> Return', '#C8E6C9'),
    (6.5, 0.5, 'Not Found\n-> Trigger Gen\nPipeline', '#FFCCBC'),
    (10.0, 1.5, 'Privacy\nFilter +\nDiversity\nRe-rank', '#D1C4E9'),
    (12.5, 1.5, 'Show to\nUser', '#FFE0B2'),
]

for x, y, text, color in serve_steps:
    box = FancyBboxPatch((x, y), 2, 1.5, boxstyle='round,pad=0.1',
                        facecolor=color, edgecolor='#333', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + 1, y + 0.75, text, ha='center', va='center', fontsize=7, fontweight='bold')

# Arrows
ax.annotate('', xy=(3.5, 2.25), xytext=(3.0, 2.25),
           arrowprops=dict(arrowstyle='->', color='#333', lw=2))
ax.annotate('', xy=(6.5, 3.0), xytext=(5.5, 2.5),
           arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.5))
ax.annotate('', xy=(6.5, 1.5), xytext=(5.5, 2.0),
           arrowprops=dict(arrowstyle='->', color='#E65100', lw=1.5))
ax.annotate('', xy=(10.0, 2.25), xytext=(8.5, 3.0),
           arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
ax.annotate('', xy=(12.5, 2.25), xytext=(12.0, 2.25),
           arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Timing annotations
ax.text(2.5, 5.3, 'Runs every 7 days (established users) / daily (new users)',
        fontsize=9, style='italic', color='#1565C0')
ax.text(2.5, 0.2, 'Real-time: <100ms end-to-end',
        fontsize=9, style='italic', color='#2E7D32')

plt.tight_layout()
plt.show()

In [ ]:
# === Serving optimization analysis ===

print("SERVING OPTIMIZATION ANALYSIS")
print("=" * 55)

# Scale numbers
total_users = 1_000_000_000   # 1B
dau = 300_000_000             # 300M DAU
avg_connections = 1_000
avg_fof = avg_connections * avg_connections  # 1M

print(f"\nScale:")
print(f"  Total users:       {total_users:>15,}")
print(f"  DAU:               {dau:>15,}")
print(f"  Avg connections:   {avg_connections:>15,}")
print(f"  Avg FoF per user:  {avg_fof:>15,}")

print(f"\nComputation Cost (per user):")
print(f"  Naive (all users):     {total_users:>12,} comparisons")
print(f"  With FoF:              {avg_fof:>12,} comparisons")
print(f"  Reduction:             {total_users/avg_fof:>12,.0f}x")

print(f"\nBatch Strategy:")
print(f"  Established users: re-compute every 7 days")
print(f"  New users (<30 days): re-compute daily")
print(f"  Inactive users: skip entirely (80% savings!)")
print(f"")
print(f"  Pre-compute ~100 suggestions per user")
print(f"  Show ~10 per visit (unseen ones only)")
print(f"  This means suggestions last ~10 visits before refresh")

print(f"\nWhy batch over online?")
print(f"  1. Social graphs are STABLE (connections barely change day-to-day)")
print(f"  2. GNN inference for 300M users in real-time is too expensive")
print(f"  3. Pre-computed results are served <10ms from database")

## Interview Cheat Sheet: Ranking, Privacy, and Serving

### Key Phrases to Drop

- "We use a **two-stage ranking** pipeline: lightweight ranker reduces FoF from 1M to 1K, then the full GNN model scores the final 50."
- "The re-ranking service adds **diversity** using MMR (Maximal Marginal Relevance) to avoid showing 10 colleagues from the same team."
- "**Privacy is non-negotiable:** we never reveal profile viewers, always respect blocks bidirectionally, and require explicit consent for phone contact matching."
- "For **cold-start users**, we fall back to attribute matching (same school, company, city) and gradually transition to graph features as they form connections."
- "We use **batch prediction** because social graphs are stable -- pre-compute every 7 days for established users, daily for new users."
- "The **north star metric** is connection requests accepted, not just sent, because a connection only forms when both sides agree."

### System Design Summary

```
1B users -> FoF (2-hop BFS) -> 1M candidates
  -> Lightweight ranker -> 1K
  -> Full model (GNN) -> 50
  -> Privacy filter -> Remove blocked/hidden/spam
  -> Diversity re-ranking -> Final 10
  -> Store in DB (batch every 7 days)
  -> Serve instantly when user opens app
```

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)